# 01 — Exploratory Data Analysis: Titanic Passenger Survival

**Goal**: Explore the Titanic dataset (891 passengers), investigate demographic signals influencing survival (`Sex`, `Pclass`, `Age`, `Fare`, `FamilySize`), identify missing value patterns, and uncover feature engineering opportunities for binary classification.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


# Resolve project root dynamically
def get_project_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "configs" / "config.yaml").exists():
            return p
    return Path.cwd().resolve()


ROOT = get_project_root()
sys.path.insert(0, str(ROOT))
from src.paths import resolve

df = pd.read_csv(resolve("data/raw/titanic.csv"))
print("Dataset shape:", df.shape)
df.head()

## 1. Target Variable: Survival Distribution & The Accuracy Trap
- Target is `Survived` (`0` = Died, `1` = Survived).
- If a dummy model predicts everyone dies (`0`), what accuracy does it achieve?
- This is why **Accuracy** alone is deceptive for classification!

In [ ]:
counts = df["Survived"].value_counts()
pcts = df["Survived"].value_counts(normalize=True) * 100
print(f"Died (0):     {counts[0]} ({pcts[0]:.1f}%)")
print(f"Survived (1): {counts[1]} ({pcts[1]:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Died (0)", "Survived (1)"],
    counts.values,
    color=["#e74c3c", "#2ecc71"],
    edgecolor="black",
)
ax.set_ylabel("Passenger Count")
ax.set_title("Target Distribution: Survived vs Died")
for bar in bars:
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        yval + 10,
        f"{yval} ({yval / len(df) * 100:.1f}%)",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
plt.grid(axis="y", alpha=0.3)
plt.show()

## 2. Missing Value Audit & Leakage Traps
- `PassengerId`: Sequential ID (Leakage/memorization trap — must drop).
- `Age`: ~20% missing (Needs thoughtful imputation: median or by passenger class/sex).
- `Cabin`: ~77% missing (Too sparse for direct category, but `has_cabin` or deck letter is informative).
- `Embarked`: 2 missing rows (Impute with mode 'S').

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_table = pd.DataFrame(
    {"Missing Count": missing, "Missing %": missing_pct.round(2)}
)
print(missing_table[missing_table["Missing Count"] > 0])

## 3. Strongest Demographic Drivers: Gender & Ticket Class
- Historical maritime protocol: *"Women and children first"*.
- Socioeconomic status (`Pclass`): 1st class vs 3rd class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Survival by Sex
sex_survival = df.groupby("Sex")["Survived"].mean() * 100
axes[0].bar(
    sex_survival.index,
    sex_survival.values,
    color=["#e91e63", "#2196f3"],
    edgecolor="black",
)
axes[0].set_title("Survival Rate by Gender")
axes[0].set_ylabel("Survival Rate (%)")
axes[0].set_ylim(0, 100)
for i, v in enumerate(sex_survival.values):
    axes[0].text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# Survival by Ticket Class
class_survival = df.groupby("Pclass")["Survived"].mean() * 100
axes[1].bar(
    ["1st Class", "2nd Class", "3rd Class"],
    class_survival.values,
    color=["#f39c12", "#95a5a6", "#795548"],
    edgecolor="black",
)
axes[1].set_title("Survival Rate by Ticket Class")
axes[1].set_ylabel("Survival Rate (%)")
axes[1].set_ylim(0, 100)
for i, v in enumerate(class_survival.values):
    axes[1].text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Exact Breakdown:")
print(
    df.groupby(["Pclass", "Sex"])["Survived"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "survival_rate"})
)

## 4. Age Distribution & Survival
- Were children prioritized over adults across all classes?

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(
    df[df["Survived"] == 1]["Age"].dropna(),
    bins=30,
    alpha=0.6,
    label="Survived (1)",
    color="#2ecc71",
    edgecolor="black",
)
plt.hist(
    df[df["Survived"] == 0]["Age"].dropna(),
    bins=30,
    alpha=0.5,
    label="Died (0)",
    color="#e74c3c",
    edgecolor="black",
)
plt.title("Age Distribution by Survival Outcome")
plt.xlabel("Age (Years)")
plt.ylabel("Passenger Count")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

print(
    "Children (<10 years old) survival rate:",
    f"{df[df['Age'] < 10]['Survived'].mean() * 100:.1f}%",
)

## 5. Feature Engineering Hypotheses for `src/data.py`
1. **`FamilySize`**: `SibSp + Parch + 1` (Solo travelers vs large families had lower survival; moderate families of 2-4 had highest survival).
2. **`IsAlone`**: Binary flag (`FamilySize == 1`).
3. **`Title` Extraction**: From `Name` (`Mr`, `Mrs`, `Miss`, `Master`, `Officer`, `Royalty`). Notice `Master` captures young boys!
4. **`HasCabin`**: Binary flag (`Cabin.notna()`). Passengers with recorded cabins were mostly 1st class near the upper deck.
5. **Categorical Encoding**: One-Hot / Ordinal encode `Sex` and `Embarked`.